In [14]:
import os
import torch
from torch.utils.data import Dataset
import pandas as pd

class MotionDataset(Dataset):
    def __init__(self, df: pd.DataFrame, pt_folder: str, training: bool = True,
                 max_points: int = 1000, augment_prob: float = 0.7):
        self.df = df.reset_index(drop=True)
        self.pt_folder = pt_folder
        self.training = training
        self.max_points = max_points
        self.augment_prob = augment_prob
        print(f"Initialized MotionDataset with {len(self.df)} samples. Training: {self.training}")

    def __len__(self) -> int:
        return len(self.df)

    def _normalize_coordinates(self, coords: torch.Tensor, height: float = 384.0, width: float = 512.0) -> torch.Tensor:
        coords_norm = coords.clone()
        coords_norm[..., 0] = coords_norm[..., 0] / width   # x
        coords_norm[..., 1] = coords_norm[..., 1] / height  # y
        return torch.clamp(coords_norm, -0.1, 1.1)

    def _calc_vel_acc(self, coords: torch.Tensor) -> torch.Tensor:
        """
        coords: [T, N, 2] normalized
        returns features: [T, N, 6] = [x, y, vx, vy, ax, ay]
        """
        T, N, _ = coords.shape
        velocity = torch.zeros_like(coords)
        if T > 1:
            velocity[1:] = coords[1:] - coords[:-1]

        acceleration = torch.zeros_like(coords)
        if T > 2:
            acceleration[2:] = velocity[2:] - velocity[1:-1]

        features = torch.cat([coords, velocity, acceleration], dim=-1)
        return features

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        vid_name, label = row['vid_name'], row['label']
        pt_path = os.path.join(self.pt_folder, f"{vid_name}_tracking.pt")

        try:
            data = torch.load(pt_path, map_location='cpu')
        except Exception as e:
            print(f"Error loading {pt_path}: {e}")
            # [T=10, N=max_points, 6]
            return (torch.zeros(10, self.max_points, 6), torch.tensor(0).long(), vid_name)

        seg_len, num_points = data['shape_info']

        # (T, N, 2) -> normalize to [0..1] approx (with clamping)
        pred_tracks = data['pred_tracks'].view(seg_len, num_points, 2)
        coords = self._normalize_coordinates(pred_tracks)

        input_tensor = self._calc_vel_acc(coords)  # [T, N, 6]
        return input_tensor, torch.tensor(label).long(), vid_name

In [17]:
import cv2
import numpy as np
import torch
import os

def _safe_uint_pt(x, y, W, H):
    xi, yi = int(round(x)), int(round(y))
    return (min(max(xi, 0), W-1), min(max(yi, 0), H-1))

def _subsample_points(P, max_points, seed=123):
    if P <= max_points:
        return np.arange(P)
    rng = np.random.default_rng(seed)
    return rng.choice(P, size=max_points, replace=False)
def render_motion_videos(
    features_T_N_6,
    save_dir,
    base_name,
    width=512,
    height=384,
    fps=20,
    max_points_to_draw=300,
    frame_step=1,
    # keep small scales; we also clamp below
    vel_scale=10.0,
    acc_scale=20.0,
    # NEW: hard caps in pixels
    vel_max_px=3,
    acc_max_px=4,
    # NEW: style: "tick" (short line), "dot" (point), or "arrow"
    style="tick",
    point_radius=2,
    draw_traces=True,
):
    import cv2, os, numpy as np, torch

    def _safe_uint_pt(x, y, W, H):
        xi, yi = int(round(x)), int(round(y))
        return (min(max(xi, 0), W-1), min(max(yi, 0), H-1))

    def _subsample_points(P, K, seed=123):
        if P <= K: return np.arange(P)
        rng = np.random.default_rng(seed)
        return rng.choice(P, size=K, replace=False)

    os.makedirs(save_dir, exist_ok=True)
    data = features_T_N_6.detach().cpu().float().numpy()
    T, N, C = data.shape
    assert C == 6, "Expect [x,y,vx,vy,ax,ay]"

    xy  = data[..., :2]
    vel = data[..., 2:4]
    acc = data[..., 4:6]

    xy_px  = np.stack([xy[...,0]*width,  xy[...,1]*height], axis=-1)
    vel_px = np.stack([vel[...,0]*width, vel[...,1]*height],  axis=-1)
    acc_px = np.stack([acc[...,0]*width, acc[...,1]*height],  axis=-1)

    idx = _subsample_points(N, max_points_to_draw)
    xy_px, vel_px, acc_px = xy_px[:, idx], vel_px[:, idx], acc_px[:, idx]
    P = xy_px.shape[1]

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    v_tracks  = cv2.VideoWriter(os.path.join(save_dir, f"{base_name}_tracks.mp4"),   fourcc, fps, (width, height))
    v_vel     = cv2.VideoWriter(os.path.join(save_dir, f"{base_name}_velocity.mp4"), fourcc, fps, (width, height))
    v_acc     = cv2.VideoWriter(os.path.join(save_dir, f"{base_name}_accel.mp4"),    fourcc, fps, (width, height))
    v_overlay = cv2.VideoWriter(os.path.join(save_dir, f"{base_name}_overlay.mp4"),  fourcc, fps, (width, height))

    prev_xy = None
    for t in range(0, T, frame_step):
        frame_tracks  = np.zeros((height, width, 3), dtype=np.uint8)
        frame_vel     = np.zeros((height, width, 3), dtype=np.uint8)
        frame_acc     = np.zeros((height, width, 3), dtype=np.uint8)
        frame_overlay = np.zeros((height, width, 3), dtype=np.uint8)

        pts = xy_px[t]; vts = vel_px[t]; ats = acc_px[t]

        for p in range(P):
            x, y = pts[p]
            xi, yi = _safe_uint_pt(x, y, width, height)

            # traces
            if draw_traces and prev_xy is not None:
                px, py = prev_xy[p]
                pxi, pyi = _safe_uint_pt(px, py, width, height)
                cv2.line(frame_tracks,  (pxi, pyi), (xi, yi), (128,128,128), 1)
                cv2.line(frame_overlay, (pxi, pyi), (xi, yi), (64,64,64),    1)

            # base point
            cv2.circle(frame_tracks,  (xi, yi), point_radius, (255,255,255), -1)
            cv2.circle(frame_overlay, (xi, yi), point_radius, (255,255,255), -1)

            # ---- Velocity (short tick / dot / arrow)
            vx, vy = vts[p]
            vec_v = np.array([vx, vy]) * vel_scale
            mag_v = float(np.linalg.norm(vec_v))
            if mag_v > 1e-8:
                if style in ("tick", "arrow"):
                    # clamp to vel_max_px
                    if mag_v > vel_max_px:
                        vec_v = vec_v * (vel_max_px / mag_v)
                if style == "dot":
                    cv2.circle(frame_vel,     (xi, yi), point_radius, (0,255,0), -1)
                    cv2.circle(frame_overlay, (xi, yi), point_radius, (0,255,0), -1)
                elif style == "tick":
                    end = _safe_uint_pt(x + vec_v[0], y + vec_v[1], width, height)
                    cv2.line(frame_vel,     (xi, yi), end, (0,255,0), 1)
                    cv2.line(frame_overlay, (xi, yi), end, (0,255,0), 1)
                else:  # "arrow"
                    end = _safe_uint_pt(x + vec_v[0], y + vec_v[1], width, height)
                    cv2.arrowedLine(frame_vel,     (xi, yi), end, (0,255,0), 1, tipLength=0.25)
                    cv2.arrowedLine(frame_overlay, (xi, yi), end, (0,255,0), 1, tipLength=0.25)

            # ---- Acceleration
            ax, ay = ats[p]
            vec_a = np.array([ax, ay]) * acc_scale
            mag_a = float(np.linalg.norm(vec_a))
            if mag_a > 1e-8:
                if style in ("tick", "arrow"):
                    if mag_a > acc_max_px:
                        vec_a = vec_a * (acc_max_px / mag_a)
                if style == "dot":
                    cv2.circle(frame_acc,     (xi, yi), point_radius, (0,0,255), -1)
                    cv2.circle(frame_overlay, (xi, yi), point_radius, (0,0,255), -1)
                elif style == "tick":
                    end = _safe_uint_pt(x + vec_a[0], y + vec_a[1], width, height)
                    cv2.line(frame_acc,     (xi, yi), end, (0,0,255), 1)
                    cv2.line(frame_overlay, (xi, yi), end, (0,0,255), 1)
                else:
                    end = _safe_uint_pt(x + vec_a[0], y + vec_a[1], width, height)
                    cv2.arrowedLine(frame_acc,     (xi, yi), end, (0,0,255), 1, tipLength=0.25)
                    cv2.arrowedLine(frame_overlay, (xi, yi), end, (0,0,255), 1, tipLength=0.25)

        v_tracks.write(frame_tracks)
        v_vel.write(frame_vel)
        v_acc.write(frame_acc)
        v_overlay.write(frame_overlay)
        prev_xy = pts.copy()

    for vw in (v_tracks, v_vel, v_acc, v_overlay):
        vw.release()

In [18]:
# /Users/mrinalraj/Downloads/WebDownload/Driving48/FullAnnotated1000/queries_Y4ARBzok9aU_00176.mp4


from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
pt_folder = "videosTensors1000"
df = pd.read_csv("df_exists1.csv")

def motion_collate_fn(batch):
    features, labels, video_names = zip(*batch)
    lengths = torch.tensor([s.shape[0] for s in features])
    features_padded = pad_sequence(features, batch_first=True, padding_value=0.0)
    max_len = features_padded.shape[1]
    mask = torch.arange(max_len).expand(len(lengths), max_len) >= lengths.unsqueeze(1)
    labels_tensor = torch.stack(labels)
    return features_padded, labels_tensor, mask, video_names

# --- Build loader
# df must have columns ['vid_name','label']
dataset = MotionDataset(df, pt_folder, training=False)
loader  = DataLoader(dataset, batch_size=1, shuffle=False, collate_fn=motion_collate_fn)

# --- Get one sample and render
features_padded, labels, mask, names = next(iter(loader))
name = names[0]
T_valid = (~mask[0]).sum().item()
sample_T_N_6 = features_padded[0, :T_valid]       # [T, N, 6]

out_dir = "./motion_viz"
render_motion_videos(
    sample_T_N_6,
    save_dir="./motion_viz",
    base_name=name,
    style="tick",          # or "dot" for points only
    vel_scale=10.0,        # small scale
    acc_scale=20.0,        # small scale
    vel_max_px=3,          # hard cap length
    acc_max_px=4,          # hard cap length
    point_radius=2
)
print("Saved:", os.listdir(out_dir))

Initialized MotionDataset with 7069 samples. Training: False
Saved: ['3qq031609lA_00002_overlay.mp4', '3qq031609lA_00002_accel.mp4', '3qq031609lA_00002_velocity.mp4', '3qq031609lA_00002_tracks.mp4']


In [19]:
import os
import glob
import cv2
import numpy as np

# ---------- helpers ----------
def _fit_and_pad(img, tile_w=320, tile_h=180, bg=(255,255,255)):
    """Resize img to fit inside (tile_w,tile_h) keeping aspect; pad with bg."""
    h, w = img.shape[:2]
    if h == 0 or w == 0:
        return np.full((tile_h, tile_w, 3), bg, dtype=np.uint8)
    scale = min(tile_w / w, tile_h / h)
    nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
    resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
    canvas = np.full((tile_h, tile_w, 3), bg, dtype=np.uint8)
    y0 = (tile_h - nh) // 2
    x0 = (tile_w - nw) // 2
    canvas[y0:y0+nh, x0:x0+nw] = resized
    return canvas

def _read_frame(cap, idx):
    """Seek to frame idx and read one frame; returns (success, frame)."""
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
    ok, frame = cap.read()
    return ok, frame

# ---------- core ----------
def extract_equally_spaced_frames(video_path, k=6):
    """
    Returns a list of (frame_img_bgr, frame_index, time_sec) for k equally spaced frames
    including first and last.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open: {video_path}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps is None or fps <= 0:  # fallback
        fps = 30.0

    # Some codecs don’t expose frame count reliably. Fallback: quick scan count.
    if total <= 0:
        total = 0
        while True:
            ok, _ = cap.read()
            if not ok:
                break
            total += 1
        cap.release()
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise RuntimeError(f"Reopen failed: {video_path}")

    if total <= 0:
        cap.release()
        raise RuntimeError(f"No frames detected: {video_path}")

    # Choose indices (0..total-1)
    if k <= 1:
        indices = [0]
    else:
        indices = np.linspace(0, total-1, num=k, dtype=int).tolist()

    frames = []
    for idx in indices:
        ok, frame = _read_frame(cap, idx)
        if not ok:
            # conservative fallback: try to read next available
            cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, idx-1))
            ok, frame = cap.read()
        if not ok or frame is None:
            # insert blank placeholder if still failing
            frame = np.zeros((240, 320, 3), dtype=np.uint8)
        t = float(idx) / fps
        frames.append((frame, int(idx), t))

    cap.release()
    return frames

def make_contact_sheet(frames, cols=3, tile_w=320, tile_h=180,
                       annotate=True, font_scale=0.5, thickness=1,
                       bg=(255,255,255), text_color=(0,0,0), pad=8):
    """
    frames: list of (img_bgr, frame_idx, time_sec)
    Returns a single BGR image tiled in rows x cols.
    """
    if len(frames) == 0:
        return np.full((tile_h, tile_w, 3), bg, dtype=np.uint8)
    rows = int(np.ceil(len(frames) / cols))

    # Build each tile
    tiles = []
    for img, fidx, tsec in frames:
        tile = _fit_and_pad(img, tile_w, tile_h, bg=bg)
        if annotate:
            label = f"f={fidx}  t={tsec:.2f}s"
            cv2.rectangle(tile, (0,0), (tile_w, 22), (255,255,255), -1)
            cv2.putText(tile, label, (6,16), cv2.FONT_HERSHEY_SIMPLEX,
                        font_scale, text_color, thickness, cv2.LINE_AA)
        tiles.append(tile)

    # Fill up to rows*cols with blanks to keep grid rectangular
    while len(tiles) < rows * cols:
        tiles.append(np.full((tile_h, tile_w, 3), bg, dtype=np.uint8))

    # Add padding between tiles
    grid_h = rows * tile_h + (rows + 1) * pad
    grid_w = cols * tile_w + (cols + 1) * pad
    sheet = np.full((grid_h, grid_w, 3), bg, dtype=np.uint8)

    k = 0
    for r in range(rows):
        for c in range(cols):
            y = pad + r * (tile_h + pad)
            x = pad + c * (tile_w + pad)
            sheet[y:y+tile_h, x:x+tile_w] = tiles[k]
            k += 1
    return sheet

def make_six_frame_contact_sheet(video_path, out_dir,
                                 tile_w=320, tile_h=180,
                                 annotate=True, cols=3):
    """
    Extract 6 equally spaced frames and save a 2x3 contact sheet PNG.
    Returns output image path.
    """
    os.makedirs(out_dir, exist_ok=True)
    frames = extract_equally_spaced_frames(video_path, k=6)
    sheet = make_contact_sheet(frames, cols=cols, tile_w=tile_w, tile_h=tile_h, annotate=annotate)
    base = os.path.splitext(os.path.basename(video_path))[0]
    out_path = os.path.join(out_dir, f"{base}_6frames.png")
    cv2.imwrite(out_path, sheet)
    return out_path

def process_videos_in_folder(in_dir, out_dir, patterns=("*.mp4","*.avi","*.mov","*.mkv","*.webm"),
                             tile_w=320, tile_h=180, annotate=True):
    """
    For every video in in_dir matching patterns, write <name>_6frames.png to out_dir.
    """
    paths = []
    for pat in patterns:
        paths.extend(glob.glob(os.path.join(in_dir, pat)))
    paths = sorted(paths)

    results = []
    for p in paths:
        try:
            out = make_six_frame_contact_sheet(p, out_dir, tile_w=tile_w, tile_h=tile_h, annotate=annotate)
            results.append(out)
            print(f"✓ {os.path.basename(p)} -> {os.path.basename(out)}")
        except Exception as e:
            print(f"✗ {os.path.basename(p)} failed: {e}")
    return results

In [20]:
# Single video
out_img = make_six_frame_contact_sheet(
    "/Users/mrinalraj/Downloads/WebDownload/Driving48/motion_viz/3qq031609lA_00002_tracks.mp4",
    out_dir="./contact_sheets",
    tile_w=400, tile_h=225,  # 16:9 tiles (adjust as you like)
    annotate=True,           # prints frame index & timestamp on each tile
    cols=3                   # 2 rows x 3 cols for 6 frames
)
print("Saved:", out_img)

# Whole folder
results = process_videos_in_folder(
    in_dir="/Users/mrinalraj/Downloads/WebDownload/Driving48/motion_viz",
    out_dir="./contact_sheets",
    tile_w=400, tile_h=225,
    annotate=True
)
print(f"Created {len(results)} contact sheets.")

Saved: ./contact_sheets/3qq031609lA_00002_tracks_6frames.png
✓ 3qq031609lA_00002_accel.mp4 -> 3qq031609lA_00002_accel_6frames.png
✓ 3qq031609lA_00002_overlay.mp4 -> 3qq031609lA_00002_overlay_6frames.png
✓ 3qq031609lA_00002_tracks.mp4 -> 3qq031609lA_00002_tracks_6frames.png
✓ 3qq031609lA_00002_velocity.mp4 -> 3qq031609lA_00002_velocity_6frames.png
Created 4 contact sheets.


In [21]:
import os, cv2, numpy as np

def save_six_frames_contact_sheet(
    video_path: str,
    out_dir: str,
    k: int = 6,
    cols: int = 3,
    tile_w: int = 400,
    tile_h: int = 225,
    annotate: bool = True,
    spacing: str = "time",  # "time" for equal time spacing, "frames" for equal frame spacing
):
    """
    Extracts 6 equally spaced frames from `video_path`, annotates them, saves each
    frame as PNG, and saves a 2x3 contact sheet image. Returns dict with paths.

    Outputs:
      - <base>_frame{0..5}_{frameIdx}.png
      - <base>_6frames.png
    """
    os.makedirs(out_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(video_path))[0]

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    if total <= 0:
        # fallback: count frames
        total = 0
        while True:
            ok, _ = cap.read()
            if not ok:
                break
            total += 1
        cap.release()
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise RuntimeError(f"Failed to reopen video: {video_path}")

    # --- choose indices (equal time by default)
    if k <= 1:
        indices = [0]
    else:
        if spacing == "frames":
            indices = np.linspace(0, max(0, total - 1), num=k, dtype=int).tolist()
        else:  # equal time spacing
            duration = (total - 1) / fps if total > 1 else 0.0
            times = np.linspace(0.0, duration, num=k)
            indices = np.clip(np.round(times * fps).astype(int), 0, max(0, total - 1)).tolist()

    # --- helpers
    def read_frame(idx):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok or frame is None:
            # try previous frame as fallback
            cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, int(idx) - 1))
            ok, frame = cap.read()
        return ok and frame is not None, frame

    def fit_and_pad(img, tw=tile_w, th=tile_h, bg=(255,255,255)):
        h, w = img.shape[:2]
        if h == 0 or w == 0:
            return np.full((th, tw, 3), bg, dtype=np.uint8)
        scale = min(tw / w, th / h)
        nw, nh = max(1, int(round(w * scale))), max(1, int(round(h * scale)))
        resized = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_AREA)
        canvas = np.full((th, tw, 3), bg, dtype=np.uint8)
        y0 = (th - nh) // 2
        x0 = (tw - nw) // 2
        canvas[y0:y0+nh, x0:x0+nw] = resized
        return canvas

    def annotate_bar(img, text, font_scale=0.6, thickness=1, fg=(0,0,0), bg=(255,255,255)):
        h, w = img.shape[:2]
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
        pad = 6
        bar_h = th + 2*pad
        # top bar
        img[0:bar_h, 0:w] = bg
        cv2.putText(img, text, (pad, pad + th), cv2.FONT_HERSHEY_SIMPLEX, font_scale, fg, thickness, cv2.LINE_AA)

    saved_frame_paths = []
    tiles = []

    # --- extract, annotate, save frames
    for i, idx in enumerate(indices):
        ok, frame = read_frame(idx)
        if not ok:
            frame = np.zeros((240, 320, 3), dtype=np.uint8)

        tsec = idx / fps
        tile = fit_and_pad(frame)
        if annotate:
            annotate_bar(tile, f"f={idx}   t={tsec:.2f}s")

        out_frame = os.path.join(out_dir, f"{base}_frame{i}_{idx}.png")
        cv2.imwrite(out_frame, tile)
        saved_frame_paths.append(out_frame)
        tiles.append(tile)

    cap.release()

    # --- build 2x3 contact sheet (cols=3)
    rows = int(np.ceil(len(tiles) / cols))
    pad = 8
    grid_h = rows * tile_h + (rows + 1) * pad
    grid_w = cols * tile_w + (cols + 1) * pad
    sheet = np.full((grid_h, grid_w, 3), 255, dtype=np.uint8)

    k_idx = 0
    for r in range(rows):
        for c in range(cols):
            y = pad + r * (tile_h + pad)
            x = pad + c * (tile_w + pad)
            if k_idx < len(tiles):
                sheet[y:y+tile_h, x:x+tile_w] = tiles[k_idx]
                k_idx += 1

    out_sheet = os.path.join(out_dir, f"{base}_6frames.png")
    cv2.imwrite(out_sheet, sheet)

    return {
        "frames": saved_frame_paths,   # list of 6 paths
        "contact_sheet": out_sheet,    # path to combined PNG
        "indices": indices,            # selected frame indices
    }

In [22]:
result = save_six_frames_contact_sheet(
    "/Users/mrinalraj/Downloads/WebDownload/Driving48/FullAnnotated1000/queries_3qq031609lA_00002.mp4",
    out_dir="./contact_sheetsFrame",
    k=6,
    spacing="time",   # equal time spacing
    annotate=True
)
print(result["contact_sheet"])
print(result["frames"])

./contact_sheetsFrame/queries_3qq031609lA_00002_6frames.png
['./contact_sheetsFrame/queries_3qq031609lA_00002_frame0_0.png', './contact_sheetsFrame/queries_3qq031609lA_00002_frame1_26.png', './contact_sheetsFrame/queries_3qq031609lA_00002_frame2_52.png', './contact_sheetsFrame/queries_3qq031609lA_00002_frame3_77.png', './contact_sheetsFrame/queries_3qq031609lA_00002_frame4_103.png', './contact_sheetsFrame/queries_3qq031609lA_00002_frame5_129.png']
